# ASL Preprocessing

***Things I changed for this setup:***

- added a "holdout" test set of 3 participants, never seen during CV so we can be sure it is generalizing the signs and not learning based on the participant.
- then the the remaining 18 participants are going to be split into 5 GroupKFold CV folds so we can tune the hyper parameters to achieve the highest accuracy

Also this will save the data so this is the only time this is needed to be run.

In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

DATA_DIR      = r'D:\MLproject\asl-signs'
TRAIN_CSV     = os.path.join(DATA_DIR, 'train.csv')
OUTPUT_DIR    = r'D:\MLproject\code\processed_data_New1'
TARGET_FRAMES = 30
MIN_FRAMES    = 5
# Justin's random seed
RANDOM_SEED   = 80
N_HOLDOUT_PARTICIPANTS = 3
N_CV_FOLDS    = 5

# 50 signs copied from original preproccessing
SELECTED_SIGNS = [
    'bird', 'fish', 'duck', 'frog', 'alligator', 'cat', 'dog', 'cow',
    'pig', 'tiger', 'lion', 'horse', 'wolf', 'bee', 'owl', 'goose',
    'jump', 'dance', 'blow', 'drink', 'drop', 'find', 'give', 'make',
    'cry', 'read', 'cut', 'hide', 'fall', 'ride',
    'yes', 'no', 'finish', 'open', 'close', 'up', 'down', 'fast',
    'quiet', 'wait', 'now', 'later', 'every', 'same', 'any',
    'pizza', 'boat', 'airplane', 'rain', 'snow'
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')
print(f'Classes: {len(SELECTED_SIGNS)}')

Output dir: D:\MLproject\code\processed_data_New1
Classes: 50


In [2]:
# ── Load and filter train.csv ─────────────────────────────────────────────────
train = pd.read_csv(TRAIN_CSV)
print(f'Total clips in dataset: {len(train)}')
print(f'Unique signs in dataset: {train["sign"].nunique()}')
print(f'Unique participants: {train["participant_id"].nunique()}')

# This will show us the change in amount of size, which is just the number of "videos" or mediapipe landmarks we have from each video
train = train[train['sign'].isin(SELECTED_SIGNS)].reset_index(drop=True)
print(f'\nAfter filtering to {len(SELECTED_SIGNS)} selected signs:')
print(f'  Clips: {len(train)}')
print(f'  Participants: {train["participant_id"].nunique()}')
print(f'  Avg clips/sign: {len(train)/train["sign"].nunique():.1f}')

Total clips in dataset: 94477
Unique signs in dataset: 250
Unique participants: 21

After filtering to 50 selected signs:
  Clips: 18907
  Participants: 21
  Avg clips/sign: 378.1


In [4]:
# The Google dataset has exactly 543 landmarks per frame, distributed as:
# face: 0-467, pose: 468-500, left_hand: 501-521, right_hand: 522-542
ROWS_PER_FRAME = 543
POSE_OFFSET = 468
NOSE_IDX = POSE_OFFSET + 0
L_SHOULDER_IDX = POSE_OFFSET + 11
R_SHOULDER_IDX = POSE_OFFSET + 12
LEFT_HAND_IDX  = slice(501, 522)
RIGHT_HAND_IDX = slice(522, 543)
N_POS_FEATURES = 137
N_FEATURES = 274

def load_parquet(path):
    df = pd.read_parquet(path, columns=['x', 'y', 'z'])
    n_frames = int(len(df) / ROWS_PER_FRAME)
    if n_frames == 0:
        return None

    data = df.values.reshape(n_frames, ROWS_PER_FRAME, 3).astype(np.float32)
    left = data[:, LEFT_HAND_IDX,  :]
    right = data[:, RIGHT_HAND_IDX, :]
    pose = np.stack([data[:, NOSE_IDX,:], data[:, L_SHOULDER_IDX,:], data[:, R_SHOULDER_IDX,:]], axis=1)
    
    # using nan for missing landmarks
    left  = np.nan_to_num(left,  nan=0.0)
    right = np.nan_to_num(right, nan=0.0)
    pose  = np.nan_to_num(pose,  nan=0.0)

    # Drop frames where both hands are missing
    left_missing  = np.all(left  == 0, axis=(1, 2))
    right_missing = np.all(right == 0, axis=(1, 2))
    valid = ~(left_missing & right_missing)
    left, right, pose = left[valid], right[valid], pose[valid]

    # Reject clips that are too short to represent a meaningful sign
    if len(left) < MIN_FRAMES:
        return None
    return {'left':left, 'right':right, 'pose':pose}

def canonicalize_hands(clip):
    # picks a dominant hand based on more presence frames
    left, right, pose = clip['left'], clip['right'], clip['pose']
    left_present  = np.any(left  != 0, axis=(1, 2))   # (T,)
    right_present = np.any(right != 0, axis=(1, 2))
    if right_present.sum() >= left_present.sum():
        dom, non = right, left
    else:
        dom, non = left, right
    return dom, non, pose

def to_flat_features(dom, non, pose):
    T = len(dom)
    out = np.zeros((T, N_POS_FEATURES), dtype=np.float32)
    out[:, 0:21]    = dom[:, :, 0]
    out[:, 21:42]   = dom[:, :, 1]
    out[:, 42:63]   = dom[:, :, 2]
    out[:, 63:84]   = non[:, :, 0]
    out[:, 84:105]  = non[:, :, 1]
    out[:, 105:126] = non[:, :, 2]
    # pose (9 features)
    out[:, 126:129] = pose[:, 0, :]  # nose
    out[:, 129:132] = pose[:, 1, :]  # left shoulder
    out[:, 132:135] = pose[:, 2, :]  # right shoulder
    # presence mask 
    dom_present = np.any(dom != 0, axis=(1, 2)).astype(np.float32)
    non_present = np.any(non != 0, axis=(1, 2)).astype(np.float32)
    out[:, 135] = dom_present
    out[:, 136] = non_present
    return out
    
def interpolate(seq, target=TARGET_FRAMES):
    # Resample every clip to a fixed frame count so all samples share shape (30, 126)
    T = len(seq)
    if T == target:
        return seq
    x_old = np.linspace(0, 1, T)
    x_new = np.linspace(0, 1, target)
    out = np.zeros((target, seq.shape[1]), dtype=np.float32)
    for i in range(seq.shape[1]):
        out[:, i] = np.interp(x_new, x_old, seq[:, i])
    return out


def normalize(seq):
    seq = seq.copy()

    for f in range(len(seq)):
        # ── hands ─────────────────────────────────────────────────────────────
        # dom wrist → indices (0, 21, 42); non wrist → (63, 84, 105)
        dx, dy, dz = seq[f, 0],  seq[f, 21], seq[f, 42]
        nx, ny, nz = seq[f, 63], seq[f, 84], seq[f, 105]
        dom_ok = not (dx == 0 and dy == 0 and dz == 0)
        non_ok = not (nx == 0 and ny == 0 and nz == 0)

        if dom_ok or non_ok:
            if dom_ok and non_ok:
                ref_x, ref_y, ref_z = (dx + nx) / 2, (dy + ny) / 2, (dz + nz) / 2
            elif dom_ok:
                ref_x, ref_y, ref_z = dx, dy, dz
            else:
                ref_x, ref_y, ref_z = nx, ny, nz

            seq[f, 0:21]    -= ref_x
            seq[f, 21:42]   -= ref_y
            seq[f, 42:63]   -= ref_z
            seq[f, 63:84]   -= ref_x
            seq[f, 84:105]  -= ref_y
            seq[f, 105:126] -= ref_z

            # Re-zero any hand that wasn't detected (subtraction corrupted its zeros)
            if not dom_ok:
                seq[f, 0:63] = 0
            if not non_ok:
                seq[f, 63:126] = 0

            xs = np.concatenate([seq[f, 0:21],   seq[f, 63:84]])
            ys = np.concatenate([seq[f, 21:42],  seq[f, 84:105]])
            zs = np.concatenate([seq[f, 42:63],  seq[f, 105:126]])
            mask = ~((xs == 0) & (ys == 0) & (zs == 0))
            if mask.sum() > 0:
                scale = np.sqrt(xs[mask]**2 + ys[mask]**2 + zs[mask]**2).mean()
                if scale >= 1e-6:
                    seq[f, 0:126] = seq[f, 0:126] / scale

        # ── pose (nose, L_shoulder, R_shoulder) ──────────────────────────────
        lsx, lsy, lsz = seq[f, 129], seq[f, 130], seq[f, 131]
        rsx, rsy, rsz = seq[f, 132], seq[f, 133], seq[f, 134]
        ls_ok = not (lsx == 0 and lsy == 0 and lsz == 0)
        rs_ok = not (rsx == 0 and rsy == 0 and rsz == 0)

        if ls_ok and rs_ok:
            cx, cy, cz = (lsx + rsx) / 2, (lsy + rsy) / 2, (lsz + rsz) / 2
            # shoulder width in xy (z from MediaPipe is less reliable)
            sh_w = np.sqrt((lsx - rsx) ** 2 + (lsy - rsy) ** 2)
            if sh_w >= 1e-6:
                for base in (126, 129, 132):
                    seq[f, base]     -= cx
                    seq[f, base + 1] -= cy
                    seq[f, base + 2] -= cz
                seq[f, 126:135] /= sh_w
            else:
                seq[f, 126:135] = 0
        else:
            seq[f, 126:135] = 0
        # presence mask (135, 136) is left untouched

    return seq

def add_velocity(seq):
    vel = np.zeros_like(seq)
    vel[1:] = seq[1:] - seq[:-1]
    return np.concatenate([seq, vel], axis=1)

sample_path = os.path.join(DATA_DIR, train['path'].iloc[0])
clip = load_parquet(sample_path)
dom, non, pose = canonicalize_hands(clip)
s = to_flat_features(dom, non, pose)
print(f'Sample shape: {s.shape}')
s = interpolate(s)
s = normalize(s)
s = add_velocity(s)
print(f'After all processing: {s.shape}')
print(f'NaN count: {np.isnan(s).sum()}')
print(f'Value range: [{s.min():.3f}, {s.max():.3f}]')

Sample shape: (23, 137)
After all processing: (30, 274)
NaN count: 0
Value range: [-3.013, 2.296]


In [5]:
# preprocessing starts here
X_all, y_all, groups_all = [], [], []
skipped = 0

for _, row in tqdm(train.iterrows(), total=len(train)):
    path = os.path.join(DATA_DIR, row['path'])
    clip = load_parquet(path)
    if clip is None:
        skipped += 1
        continue
    dom, non, pose = canonicalize_hands(clip)
    seq = to_flat_features(dom, non, pose)
    seq = interpolate(seq)
    seq = normalize(seq)
    seq = add_velocity(seq)
    X_all.append(seq)
    y_all.append(row['sign'])
    groups_all.append(row['participant_id'])

X_all = np.array(X_all, dtype=np.float32)
y_all = np.array(y_all)
groups_all = np.array(groups_all)

print(f'\nFinal: {X_all.shape}, labels {y_all.shape}, groups {groups_all.shape}')
print(f'Skipped: {skipped}')
print(f'Unique participants: {len(np.unique(groups_all))}')

100%|████████████████████████████████████████████████████████████████████████████| 18907/18907 [14:50<00:00, 21.22it/s]



Final: (18848, 30, 274), labels (18848,), groups (18848,)
Skipped: 59
Unique participants: 21


In [6]:
# labels
le = LabelEncoder()
y_enc = le.fit_transform(y_all)
print(f'Classes: {len(le.classes_)}')
print(f'Class names: {list(le.classes_)}')

Classes: 50
Class names: [np.str_('airplane'), np.str_('alligator'), np.str_('any'), np.str_('bee'), np.str_('bird'), np.str_('blow'), np.str_('boat'), np.str_('cat'), np.str_('close'), np.str_('cow'), np.str_('cry'), np.str_('cut'), np.str_('dance'), np.str_('dog'), np.str_('down'), np.str_('drink'), np.str_('drop'), np.str_('duck'), np.str_('every'), np.str_('fall'), np.str_('fast'), np.str_('find'), np.str_('finish'), np.str_('fish'), np.str_('frog'), np.str_('give'), np.str_('goose'), np.str_('hide'), np.str_('horse'), np.str_('jump'), np.str_('later'), np.str_('lion'), np.str_('make'), np.str_('no'), np.str_('now'), np.str_('open'), np.str_('owl'), np.str_('pig'), np.str_('pizza'), np.str_('quiet'), np.str_('rain'), np.str_('read'), np.str_('ride'), np.str_('same'), np.str_('snow'), np.str_('tiger'), np.str_('up'), np.str_('wait'), np.str_('wolf'), np.str_('yes')]


### Mostly Claude from here down, never used the unique participants setup so basically just copy and pasted to get moving onto the RF ###

In [7]:
# setting up the holdout test with the 3 participants that won't be in the training data
unique_participants = np.unique(groups_all)
print(f'Total participants: {len(unique_participants)}')

# splits evenly
holdout_frac = N_HOLDOUT_PARTICIPANTS / len(unique_participants)
gss = GroupShuffleSplit(n_splits=1, test_size=holdout_frac, random_state=RANDOM_SEED)
dev_idx, test_idx = next(gss.split(X_all, y_enc, groups=groups_all))

X_dev, y_dev, groups_dev = X_all[dev_idx],  y_enc[dev_idx],  groups_all[dev_idx]
X_test, y_test, groups_test = X_all[test_idx], y_enc[test_idx], groups_all[test_idx]

print(f'\nDev set:  {X_dev.shape}, {len(np.unique(groups_dev))} participants')
print(f'Test set: {X_test.shape}, {len(np.unique(groups_test))} participants')
print(f'Test participants: {np.unique(groups_test)}')

# double check to make sure none are contaminated
assert len(set(groups_dev) & set(groups_test)) == 0, 'participant leakage!'

Total participants: 21

Dev set:  (16007, 30, 274), 18 participants
Test set: (2841, 30, 274), 3 participants
Test participants: [27610 49445 61333]


In [8]:
# 5 cross validation folds to hep with hyper paramater tuning
gkf = GroupKFold(n_splits=N_CV_FOLDS)
fold_indices = []
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_dev, y_dev, groups=groups_dev)):
    fold_indices.append((tr_idx, va_idx))
    tr_participants = set(groups_dev[tr_idx])
    va_participants = set(groups_dev[va_idx])
    assert not (tr_participants & va_participants), f'fold {fold} leaks!'
    print(f'Fold {fold+1}: train={len(tr_idx)} clips / {len(tr_participants)} ppl, '
          f'val={len(va_idx)} clips / {len(va_participants)} ppl')

Fold 1: train=12489 clips / 14 ppl, val=3518 clips / 4 ppl
Fold 2: train=13176 clips / 15 ppl, val=2831 clips / 3 ppl
Fold 3: train=12635 clips / 14 ppl, val=3372 clips / 4 ppl
Fold 4: train=13174 clips / 15 ppl, val=2833 clips / 3 ppl
Fold 5: train=12554 clips / 14 ppl, val=3453 clips / 4 ppl


In [9]:
# ── Save everything ───────────────────────────────────────────────────────────
# The downstream notebooks just load these files.

np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'dev.npz'),
    X=X_dev, y=y_dev, groups=groups_dev
)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'test.npz'),
    X=X_test, y=y_test, groups=groups_test
)
# Save fold indices so both models use identical splits
np.savez(
    os.path.join(OUTPUT_DIR, 'cv_folds.npz'),
    **{f'fold_{i}_train': tr for i, (tr, _) in enumerate(fold_indices)},
    **{f'fold_{i}_val':   va for i, (_, va) in enumerate(fold_indices)},
)
# Save label encoder classes
np.save(os.path.join(OUTPUT_DIR, 'classes.npy'), le.classes_)

print('Saved:')
for f in ['dev.npz', 'test.npz', 'cv_folds.npz', 'classes.npy']:
    p = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}: {os.path.getsize(p)/1024/1024:.1f} MB')

Saved:
  dev.npz: 311.9 MB
  test.npz: 51.5 MB
  cv_folds.npz: 0.6 MB
  classes.npy: 0.0 MB
